In [2]:
pip install rebound

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install reboundx

Note: you may need to restart the kernel to use updated packages.


In [46]:
import rebound
import reboundx
import numpy as np
import matplotlib.pyplot as plt
from pycbc.waveform import get_td_waveform
from gwpy.timeseries import TimeSeries
from gwosc.datasets import event_gps

In [48]:
pip install gwpy

Note: you may need to restart the kernel to use updated packages.


In [20]:
import numpy as np
import matplotlib.pyplot as plt
import rebound
import reboundx
from scipy.signal import correlate, hilbert, butter, filtfilt
from gwpy.timeseries import TimeSeries
from scipy.signal.windows import tukey # Import the windowing function


# --- 1. CONFIGURATION FOR GW150914 ---
# The masses must match the event to get the correct frequency evolution (chirp)
M1 = 30.0  # Solar masses
M2 = 34.6  # Solar masses
DISTANCE_MPC = 470 # Approx distance for amplitude scaling
# LIGO event time
GPS_TIME = 1126259462.4

MPC_TO_M = 3.086e22

# GET AND PROCESS LIGO DATA
print("Fetching LIGO Data for GW150914...")
# Fetch 32 seconds of data around the event
data = TimeSeries.fetch_open_data('H1', GPS_TIME - 16, GPS_TIME + 16)

# A. WHITENING
white_data = data.whiten(4, 2) # 4s segment, 2s overlap

# B. BANDPASS FILTERING
# Focus on the 30Hz - 400Hz band where the signal exists
bp_data = white_data.bandpass(30, 400)
ligo_strain = bp_data.value

Fetching LIGO Data for GW150914...


In [21]:
import numpy as np
from pycbc.waveform import get_td_waveform
from pycbc.filter import matched_filter

def matched_filtering(M_1, M_2, e, distance, ligo_data):
    MPC_TO_M = 3.086e22
    distance *= MPC_TO_M
    h_p, h_c = get_td_waveform(approximant="EccentricTD", 
                         mass1=M_1,                   
                         mass2=M_2,                   
                         spin1z=0, spin2z=0,         
                         eccentricity=e,           
                         delta_t=1.0/4096,           
                         f_lower=1.0)
    h_plus_1d = h_p

    
    # Apply same bandpass to template
    # Create a TimeSeries object for easier filtering
    template_ts = TimeSeries(h_plus_1d, sample_rate=4096)
    template_bp = template_ts.bandpass(30, 400)
    
    # Convert to numpy arrays
    ligo_strain = ligo_data.value
    sim_strain = template_bp.value
    
    seconds_to_keep = 0.3
    samples_to_keep = int(seconds_to_keep * 4096) 
    short_sim_strain = template_bp.value[-samples_to_keep:] 
    
    # 2. Taper the edges to prevent artifacts
    window = tukey(len(short_sim_strain), alpha=0.1)
    short_sim_strain_tapered = short_sim_strain * window
    
    # 3. Create analytic template and normalize
    analytic_template = hilbert(short_sim_strain_tapered)
    analytic_template -= np.mean(analytic_template) 
    analytic_template /= np.linalg.norm(analytic_template) 
    
    # 4. Cross Correlate
    cc = correlate(ligo_strain, analytic_template, mode='same')
    abs_cc = np.abs(cc)
    match_score = np.max(abs_cc)
    return match_score

In [22]:
matched_filtering(30.0, 34.6, 0.000001, 470, bp_data)

5.7943189077150254

In [30]:
import numpy as np

M_1 = 30.0
M_2 = 34.6
distance = 470

def intial_eccentricity(M_1, M_2, distance, ligo_data):
    # Initial search parameters
    e = [0.000000000001, 0.1, 0.2] # Your starting 'e' list
    iterations = 3  # How deep to go (3 = down to 0.001 precision)
    
    print(f"Starting Search with grid: {e}\n")
    
    for depth in range(iterations):
        print(f"--- Iteration {depth + 1} ---")
        
        # 1. Calculate scores for the current grid
        scores = []
        for e_val in e:
            # Check if matched_filtering is returning a single value or array
            score = matched_filtering(M_1, M_2, e_val, distance, ligo_data)
            scores.append(score)
        
        # 2. Find the best match
        best_score = np.max(scores)
        best_idx = scores.index(best_score)
        best_e = e[best_idx]
        
        print(f"Best e at this depth: {best_e:.5f} (Index: {best_idx})")
    
        # 3. Define the next search range
        
        start_e = 0
        end_e = 0
        
        # CASE A: Best match is at the very START of the list
        if best_idx == 0:
            print(" > Peak is at the lower boundary.")
            start_e = e[0]
            end_e = e[1]
    
        # CASE B: Best match is at the very END of the list
        elif best_idx == len(e) - 1:
            print(" > Peak is at the upper boundary.")
            start_e = e[-2]
            end_e = e[-1]
    
        # CASE C: Best match is in the MIDDLE
        else:
            # Compare neighbors to decide which side to drill down into
            score_left = scores[best_idx - 1]
            score_right = scores[best_idx + 1]
            
            if score_left > score_right:
                start_e = e[best_idx - 1]
                end_e = e[best_idx]
            else:
                start_e = e[best_idx]
                end_e = e[best_idx + 1]
    
        print(f" > Refining search between {start_e:.5f} and {end_e:.5f}")
    
        # 4. Generate the new finer grid
        e = np.linspace(start_e, end_e, 11)
        
    print(f"\nFinal estimated eccentricity: {best_e}")

In [32]:
def retrieve_data(bh_merger):
    gps = event_gps(bh_merger)
    start = gps - 16.0
    end = gps + 16.0
    return start, end

In [38]:
from gwpy.table import EventTable

def fetch_data(bh_merger_event):
    events = EventTable.fetch_open_data(
        "GWTC-1-confident", 
        columns=["name", "mass_1_source", "mass_2_source", "luminosity_distance"]
    )
        
    data = events[events['name'] == bh_merger_event + '-v3']
    
    m1 = data['mass_1_source'][0]
    m2 = data['mass_2_source'][0]
    dist = data['luminosity_distance'][0]
    
    return m1, m2, dist

In [40]:
m1, m2, dist = fetch_data("GW170608")
fetch_data("GW170608")

(11.0, 7.6, 320.0)

In [52]:
def find_initial_eccentricity(bh_merger_event):
    start, end = retrieve_data(bh_merger_event)
    M_1, M_2, dist = fetch_data(bh_merger_event)
    # Fetch 32 seconds of data around the event
    data = TimeSeries.fetch_open_data('H1', start, end)
    
    # A. WHITENING
    # Whitening normalizes the noise power across frequencies
    white_data = data.whiten(4, 2) # 4s segment, 2s overlap
    
    # B. BANDPASS FILTERING
    # Focus on the 30Hz - 400Hz band where the signal exists
    bp_data = white_data.bandpass(30, 400)
    
    intial_eccentricity(M_1, M_2, dist, bp_data)

In [50]:
find_initial_eccentricity("GW150914")

Starting Search with grid: [1e-12, 0.1, 0.2]

--- Iteration 1 ---
Best e at this depth: 0.00000 (Index: 0)
 > Peak is at the lower boundary.
 > Refining search between 0.00000 and 0.10000
--- Iteration 2 ---
Best e at this depth: 0.07000 (Index: 7)
 > Refining search between 0.07000 and 0.08000
--- Iteration 3 ---
Best e at this depth: 0.07500 (Index: 5)
 > Refining search between 0.07400 and 0.07500

Final estimated eccentricity: 0.07500000000025
